## Bussiness case:

We’re opening a chocolate business and want to study our biggest competitor, from which we have access to their sales.

[Link for dataset](https://www.kaggle.com/datasets/arjunmehta1992/chocolate-sales-in-20222023/data)

We will look at different data to compare and prepare our go to market

We will shape the data in 3 tables: invoices, salesperson, product

We want to find out:

- Their prices
- Their discounts
- Seasonality of their sales
- Best salesperson to recruit for our company
- What country buys the most?

Viz:
- Leaderboard of sales
- Leaderboard countries

## About Dataset

This dataset captures order-level sales activity for a chocolate brand operating across multiple countries and sales channels between January 2022 and December 2023. Each row represents a single sales order, covering product details, discounting, marketing spend, and the resulting shipment volume.

The data reflects a mix of retail, online, and wholesale channels, with seasonal demand patterns (notably around Valentine's Day and the year-end holiday season) and variation across 25 salespeople, 7 countries, and 12 product lines.

As with most real-world sales exports, the data contains some inconsistencies — missing fields, duplicate records, mixed date formats, and a handful of outliers — that came through during the export/merge process and were left as-is.

Content

    Order_ID — Unique identifier for each order

    Product — Chocolate product name

    Country — Country of sale

    Channel — Sales channel (Retail / Online / Wholesale)

    Salesperson — Sales representative handling the order

    Order_Date — Date the order was placed

    Discount_Pct — Discount applied to the order (%)

    Price_per_Box — Price per box after discount (USD)

    Marketing_Spend — Marketing budget allocated around the order (USD)

    Boxes_Shipped — Number of boxes shipped for the order

    Amount — Total order value (Boxes_Shipped × Price_per_Box)


## Loading libraries

In [1]:
import pandas as pd
import numpy as np
import cleaning_chocolate as cc

%load_ext autoreload
%autoreload 2

## Loading dataframe

In [2]:
chocolate_df = pd.read_csv('Chocolate_Sales.csv')
chocolate_df

,Order_ID,Product,Country,Channel,Salesperson,Order_Date,Discount_Pct,Price_per_Box,Marketing_Spend,Boxes_Shipped,Amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


## Dataset issues:

### Missing values in the dataset:

order_date 437 rows -> drop as is 1% of the total rows

discount_pct 489 -> means no discount (add 0)

price_per_box 457 -> perhaps the values are somewhere else

marketing_spend 461 -> means no money spent (add 0)

#### Amount doesn't match price_per_box * boxes shipped. When adding the discount back it seems similar
- Fix: create a new amount column price_per_box * boxes_shipped
- Fix: rename old amount to amount_deprecated
- Fix: create a whole amount per box column (price per box+discount)

### Price per box is with price discounted. 
- Fix: Add back discount to new column to know what is the original price

### Fix box shipped when they're on minus
- Fix: They are returns





## Cleaning

In [3]:
#First, let's clean the column names of the chocolate DataFrame using the `clean_column_names` function from the 
# `cleaning_chocolate` module.

cleaned_chocolate_df = cc.clean_column_names(chocolate_df)
cleaned_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


In [4]:
# Drop rows with missing values in the 'order_date' column as they are just 1% of the rows.
cleaned_chocolate_df = cc.drop_blank_order_dates(cleaned_chocolate_df)


In [26]:
# Fill missing values in the 'discount_pct' and 'marketing_spend' columns with 0.
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'discount_pct')
cleaned_chocolate_df = cc.fill_missing_with_zero(cleaned_chocolate_df, 'marketing_spend')
cleaned_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box,marketing_spend,boxes_shipped,amount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69
...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51


In [6]:
# Create price columns (before/after discount)

# Create a working copy to avoid modifying the original cleaned DataFrame
fixed_chocolate_df = cleaned_chocolate_df.copy()

In [ ]:
# Create price columns (before/after discount)

# Create a working copy to avoid modifying the original cleaned DataFrame
fixed_chocolate_df = cleaned_chocolate_df.copy()

In [27]:
# Generate price columns: adds 'price_per_box_before_discount' and renames
# 'price_per_box' → 'price_per_box_after_discount'

fixed_chocolate_df = cc.create_price_columns(fixed_chocolate_df)

In [28]:
# Inspect the new price columns to verify the discount calculation logic

fixed_chocolate_df[[
    "discount_pct",
    "price_per_box_after_discount",
    "price_per_box_before_discount"
]].head(10).round(2)

,discount_pct,price_per_box_after_discount,price_per_box_before_discount
0,3.5,13.72,14.22
1,9.4,3.30,3.64
2,4.9,18.21,19.15
3,15.0,2.66,3.13
4,4.4,2.75,2.88
5,17.8,12.87,15.66
6,12.0,3.18,3.61
7,22.6,2.43,3.14
8,13.9,2.98,3.46
9,16.3,3.13,3.74


In [29]:
# Now let's validate price calculation logic

# Create a boolean mask for rows where 'price_per_box_after_discount' is not null

valid_prices = fixed_chocolate_df["price_per_box_after_discount"].notna()


# Assert that price before discount is always >= price after discount
# (should return True if the calculation is correct)

(
    fixed_chocolate_df.loc[
        valid_prices, "price_per_box_before_discount"
    ]
    >=
    fixed_chocolate_df.loc[
        valid_prices, "price_per_box_after_discount"
    ]
).all()

np.True_

In [30]:
# Create amount columns (order totals and discounts)

fixed_chocolate_df = cc.create_amount_columns_without_total(fixed_chocolate_df)

In [31]:
# Display a sample of all relevant price and amount columns to verify the output

fixed_chocolate_df[[
    "discount_pct",
    "price_per_box_before_discount",
    "price_per_box_after_discount",
    "boxes_shipped",
    "amount_before_discount",
    "amount_after_discount",
    "amount_deprecated"
]].head(10)

,discount_pct,price_per_box_before_discount,price_per_box_after_discount,boxes_shipped,amount_before_discount,amount_after_discount,amount_deprecated
0,3.5,14.217617,13.72,71,1009.45,974.12,912.31
1,9.4,3.642384,3.30,84,305.96,277.20,245.91
2,4.9,19.148265,18.21,35,670.19,637.35,583.7
3,15.0,3.129412,2.66,92,287.91,244.72,211.27
4,4.4,2.876569,2.75,214,615.59,588.50,549.69
5,17.8,15.656934,12.87,47,735.88,604.89,527.01
6,12.0,3.613636,3.18,159,574.57,505.62,470.63
7,22.6,3.139535,2.43,326,1023.49,792.18,629.51
8,13.9,3.461092,2.98,70,242.28,208.60,181.16
9,16.3,3.739546,3.13,213,796.52,666.69,588.18


In [12]:
fixed_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box_after_discount,marketing_spend,boxes_shipped,amount_deprecated,price_per_box_before_discount,amount_before_discount,amount_after_discount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31,14.217617,1009.45,974.12
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91,3.642384,305.96,277.20
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7,19.148265,670.19,637.35
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27,3.129412,287.91,244.72
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69,2.876569,615.59,588.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56,3.791946,473.99,423.75
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47,3.712213,549.41,454.36
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79,3.353733,499.71,408.26
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51,15.766262,378.39,343.20


In [32]:
# There are some missing values in the 'amount_after_discount' column, which is calculated as 'amount_before_discount' - 'total_discount'.
# We are going to fill them with the median per product, then compute total_discount from the filled values.

cols_to_impute = [
    "price_per_box_after_discount",
    "price_per_box_before_discount",
    "amount_before_discount",
    "amount_after_discount",
]

fixed_chocolate_df = cc.impute_missing_by_group(fixed_chocolate_df, cols_to_impute)
fixed_chocolate_df = cc.compute_total_discount(fixed_chocolate_df)

In [33]:
# Now all the code should be fixed and the missing values filled. Let's check the DataFrame again.
fixed_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box_after_discount,marketing_spend,boxes_shipped,amount_deprecated,price_per_box_before_discount,amount_before_discount,amount_after_discount,total_discount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31,14.217617,1009.45,974.12,35.33
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91,3.642384,305.96,277.20,28.76
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7,19.148265,670.19,637.35,32.84
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27,3.129412,287.91,244.72,43.19
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69,2.876569,615.59,588.50,27.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56,3.791946,473.99,423.75,50.24
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47,3.712213,549.41,454.36,95.05
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79,3.353733,499.71,408.26,91.45
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51,15.766262,378.39,343.20,35.19


In [35]:
# Convert order_date from string to datetime so it exports as a proper DATE column for MySQL rather than text.

fixed_chocolate_df = cc.convert_order_date_dtype(fixed_chocolate_df)

## Creating tables

## Products

In [15]:
# Now that we have cleaned the data, let's create a new DataFrame with one row per product, containing the median price per box before discount for each product.

product_df = (
    fixed_chocolate_df[["product", "price_per_box_before_discount"]]
    .rename(columns={
        "product": "product_name",
        "price_per_box_before_discount": "price"
    })
    .groupby("product_name", as_index=False)["price"]
    .median().round(2)
)

product_df


,product_name,price
0,70% Dark Bar,3.80
1,85% Dark Bar,3.81
2,Almond Crunch Bar,3.80
3,Choco Coins Bag,3.79
4,Hazelnut Milk Bar,3.80
5,Milk Classic Bar,3.79
6,Mint Dark Bar,3.78
7,Mixed Assortment Box,3.79
8,Pralines Gift Box,3.79
9,Salted Caramel Bar,3.81


Products have a very similar price, let's explore further

In [16]:
# Check whether price_per_box varies meaningfully by product, country, channel
group_stats = (
    fixed_chocolate_df
    .groupby(["product", "country", "channel"])["price_per_box_before_discount"]
    .agg(["median", "mean", "std", "count"])
)

overall_mean = fixed_chocolate_df["price_per_box_before_discount"].mean()

between_group_var = (
    (group_stats["mean"] - overall_mean) ** 2 * group_stats["count"]
).sum() / group_stats["count"].sum()

within_group_var = (
    group_stats["std"] ** 2 * (group_stats["count"] - 1)
).sum() / (group_stats["count"].sum() - len(group_stats))

print(f"Number of product x country x channel groups: {len(group_stats)}")
print(f"Group median range: {group_stats['median'].min():.2f} - {group_stats['median'].max():.2f}")
print(f"Between-group variance: {between_group_var:.4f}")
print(f"Within-group variance:  {within_group_var:.4f}")
print(f"Ratio (between/within): {between_group_var / within_group_var:.4f}")

Number of product x country x channel groups: 180
Group median range: 3.52 - 4.07
Between-group variance: 0.0307
Within-group variance:  32.8451
Ratio (between/within): 0.0009


**Finding:** even at the finest cut (product × country × channel, 180 groups),
median prices only range ~3.5–4.1, and the between-group variance is roughly
1000x smaller than the within-group variance. This means product, country,
and channel explain almost none of the variation in price — `price_per_box`
behaves as if it were generated independently of these dimensions in the
source data. We therefore treat `price_per_box_before_discount` as an
algebraically reconstructed field rather than a recoverable "true" catalog
price, and do not attempt further segmentation.

In [17]:
# One row per product, with the current price stats for reference, just in case we have missed something.
product_prices = (
    fixed_chocolate_df
    .groupby("product")["price_per_box_before_discount"]
    .agg(["median", "mean", "min", "max", "std", "count"])
    .round(2)
    .reset_index()
)

product_prices

,product,median,mean,min,max,std,count
0,70% Dark Bar,3.80,6.84,2.07,26.34,5.74,64673
1,85% Dark Bar,3.81,6.83,2.06,24.49,5.66,7985
2,Almond Crunch Bar,3.80,6.80,2.18,27.06,5.72,9166
3,Choco Coins Bag,3.79,6.83,2.22,26.12,5.79,6455
4,Hazelnut Milk Bar,3.80,6.74,2.32,25.11,5.67,5318
5,Milk Classic Bar,3.79,6.82,2.09,29.05,5.74,16143
6,Mint Dark Bar,3.78,6.80,2.04,26.15,5.75,10566
7,Mixed Assortment Box,3.79,6.84,2.20,26.03,5.78,21415
8,Pralines Gift Box,3.79,6.71,2.24,25.94,5.64,12702
9,Salted Caramel Bar,3.81,6.92,2.46,25.14,5.78,5789


The product price doesn't have sense as it has been provided in the original database. In that case, we will fix the prices according to market standards.

In [36]:
new_prices = {
    "70% Dark Bar": 4.50,
    "85% Dark Bar": 5.00,
    "Almond Crunch Bar": 4.25,
    "Choco Coins Bag": 3.00,
    "Hazelnut Milk Bar": 4.75,
    "Milk Classic Bar": 3.50,
    "Mint Dark Bar": 4.60,
    "Mixed Assortment Box": 9.50,
    "Pralines Gift Box": 8.50,
    "Salted Caramel Bar": 5.20,
    "Truffle Gift Box": 8.00,
    "White Choco Bar": 4.10,
}

Let's create the final products table now:

In [37]:
# One row per product, surrogate product_id, and the market-standard
# reference price mapped in from new_prices.

products = cc.build_products_table(fixed_chocolate_df, new_prices)
products

,product_id,product,price
0,1,70% Dark Bar,4.50
1,2,85% Dark Bar,5.00
2,3,Almond Crunch Bar,4.25
3,4,Choco Coins Bag,3.00
4,5,Hazelnut Milk Bar,4.75
5,6,Milk Classic Bar,3.50
6,7,Mint Dark Bar,4.60
7,8,Mixed Assortment Box,9.50
8,9,Pralines Gift Box,8.50
9,10,Salted Caramel Bar,5.20


## Salesperson table

In [38]:
# One row per unique salesperson, with a surrogate salesperson_id
salespersons = cc.build_salespersons_table(fixed_chocolate_df)
salespersons

,salesperson_id,salesperson
0,1,Arjun Mehta
1,2,Hannah Müller
2,3,Yuki Sato
3,4,Lucas Walker
4,5,Ananya Gupta
5,6,Rohan Patel
6,7,Priya Sharma
7,8,Carlos Mendez
8,9,Camila Santos
9,10,Emily Clarke


## Invoices table

fixed_chocolate_df

In [21]:
fixed_chocolate_df

,order_id,product,country,channel,salesperson,order_date,discount_pct,price_per_box_after_discount,marketing_spend,boxes_shipped,amount_deprecated,price_per_box_before_discount,amount_before_discount,amount_after_discount,total_discount
0,ORD-069833,Truffle Gift Box,Australia,Retail,Arjun Mehta,2022-12-11,3.5,13.72,202.03,71,912.31,14.217617,1009.45,974.12,35.33
1,ORD-090726,85% Dark Bar,Australia,Retail,Arjun Mehta,2023-03-14,9.4,3.30,55.18,84,245.91,3.642384,305.96,277.20,28.76
2,ORD-042159,70% Dark Bar,Japan,Retail,Hannah Müller,2023-12-21,4.9,18.21,60.65,35,583.7,19.148265,670.19,637.35,32.84
3,ORD-197166,Hazelnut Milk Bar,Germany,Retail,Arjun Mehta,2023-12-18,15.0,2.66,52.00,92,211.27,3.129412,287.91,244.72,43.19
4,ORD-112162,Almond Crunch Bar,Australia,Retail,Yuki Sato,2023-08-18,4.4,2.75,187.44,214,549.69,2.876569,615.59,588.50,27.09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,ORD-053460,70% Dark Bar,Australia,Retail,Beatriz Souza,2022-05-27,10.6,3.39,186.19,125,418.56,3.791946,473.99,423.75,50.24
199996,ORD-010743,Truffle Gift Box,India,Wholesale,Arjun Mehta,2022-09-06,17.3,3.07,42.28,148,366.47,3.712213,549.41,454.36,95.05
199997,ORD-049690,Salted Caramel Bar,Japan,Online,Arjun Mehta,2022-06-16,18.3,2.74,100.19,149,329.79,3.353733,499.71,408.26,91.45
199998,ORD-189637,85% Dark Bar,Japan,Wholesale,Rohan Patel,2022-10-14,9.3,14.30,29.96,24,304.51,15.766262,378.39,343.20,35.19


In [39]:
# Build the invoices table. Price/amount columns are recalculated from the
# market-standard product price (products.price), not the original
# price_per_box, since that field was confirmed to be randomly generated.

invoices = cc.build_invoices_table(fixed_chocolate_df, products, salespersons)
invoices

,order_id,product_id,salesperson_id,country,channel,order_date,discount_pct,marketing_spend,boxes_shipped,price_per_box_before_discount,price_per_box_after_discount,amount_before_discount,amount_after_discount,total_discount
0,ORD-069833,11,1,Australia,Retail,2022-12-11,3.5,202.03,71,8.00,7.72,568.0,548.12,19.88
1,ORD-090726,2,1,Australia,Retail,2023-03-14,9.4,55.18,84,5.00,4.53,420.0,380.52,39.48
2,ORD-042159,1,2,Japan,Retail,2023-12-21,4.9,60.65,35,4.50,4.28,157.5,149.80,7.70
3,ORD-197166,5,1,Germany,Retail,2023-12-18,15.0,52.00,92,4.75,4.04,437.0,371.68,65.32
4,ORD-112162,3,3,Australia,Retail,2023-08-18,4.4,187.44,214,4.25,4.06,909.5,868.84,40.66
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199558,ORD-053460,1,11,Australia,Retail,2022-05-27,10.6,186.19,125,4.50,4.02,562.5,502.50,60.00
199559,ORD-010743,11,1,India,Wholesale,2022-09-06,17.3,42.28,148,8.00,6.62,1184.0,979.76,204.24
199560,ORD-049690,10,1,Japan,Online,2022-06-16,18.3,100.19,149,5.20,4.25,774.8,633.25,141.55
199561,ORD-189637,2,6,Japan,Wholesale,2022-10-14,9.3,29.96,24,5.00,4.54,120.0,108.96,11.04
